# 04 — Spatial-scale sensitivity and forest-structure dependence

**Main scientific contribution for Nadine's crown-segmentation paper:** Is SAM1 fit for estimating ecological forest structure, and at what observation scale? Distinguish aggregate agreement caused by cancellation from genuine spatial segmentation accuracy.

**Frozen inputs:** six Baego ROIs, Kristof reference crowns (≥5 m²) and consistently inferred SAM1 ViT-H + NMS polygons (with exterior halo). No model fitting, inference or retraining.

### Analysis A — Strict paired aggregation (primary)

Within every complete 40 × 40 m parent window, evaluate **the same 1,600 m² footprint** as sixteen 10 × 10 m subwindows, four 20 × 20 m subwindows, and one 40 × 40 m parent. For each parent, average the *absolute* canopy-cover errors at fine scales and compare with error after aggregation. Test whether spatial disagreement remains unchanged despite error cancellation. **Do not conflate this with 80 m or full-ROI estimates**, which use different footprints and remain descriptive.

### Analysis B — Forest-structure dependence (exploratory)

Relate errors to reference crown density, size, canopy cover, heterogeneity and nearest-neighbour crowding, focusing on **20 m** windows and a supplementary 40 m analysis. Reuse Notebook 03 crown-level results when available.

**Caveats:** Only six ROIs; within-ROI windows are spatially correlated. Treat ROI (not windows) as the primary unit for interpretation. These polygons are a frozen RGB SAM1 baseline; results should ultimately be reproduced with the manuscript's final SAM-frst output.

**Outputs:** `V2/Crown/scale_structure_analysis/`, with the original Notebook 04 files preserved and additional `04_paired_*` outputs.

## 01 — Setup, data paths and frozen analysis settings

In [ ]:
# 01 — CPU setup; consistent inputs from Notebooks 00–03
!pip -q install geopandas shapely scipy rasterio

from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import rasterio, matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union
from shapely import make_valid
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from google.colab import drive

drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
DATA=ROOT/"data/Geo_Ref"
RGB=DATA/"Baego_georef.tif"
ROI_FILE=DATA/"Annotations/ROI_Crowns_Kvo.shp"
REF_FILE=DATA/"Annotations/Crown_Kristof.shp"
SAM_DIR=ROOT/"V2/Crown/sam1_core_centroid_halo40_v2"
OUT=ROOT/"V2/Crown/scale_structure_analysis"
ROIS=tuple(range(6))
METHOD="nms_060"
SCALES_M=(10,20,40,80)
REFERENCE_MIN_AREA=5.0
# Full square windows entirely inside each irregular ROI; no fake 80m samples
MIN_INSIDE_FRACTION=0.999
for path in (RGB,ROI_FILE,REF_FILE,SAM_DIR):
    if not path.exists():raise FileNotFoundError(path)
for roi in ROIS:
    if not (SAM_DIR/f"roi{roi}_{METHOD}.gpkg").is_file():
        raise FileNotFoundError(SAM_DIR/f"roi{roi}_{METHOD}.gpkg")
OUT.mkdir(parents=True,exist_ok=True)
with rasterio.open(RGB) as ds:
    CRS, BOUNDS=ds.crs,ds.bounds
assert CRS is not None and CRS.is_projected
print("Evaluation window sizes (m):",SCALES_M)
print("ROI total areas and methods are independent of the SAM 40m inference core.")
print("Output:",OUT)

## 02 — Projected polygons, reference QA and six-ROI inventory

Reference crowns ≥5 m². Crown centroids and predicted polygons share the Baego raster CRS. Missing reference .prj is assigned only if raw coordinates overlap the raster. The 40 m exterior inference halo is preserved for boundary-crossing canopy cover.

In [ ]:
# 02 — Geometry normalization; load once
def polygonal(g):
    if g is None or g.is_empty:return Polygon()
    if not g.is_valid:g=make_valid(g)
    if isinstance(g,(Polygon,MultiPolygon)):return g
    parts=[a for a in getattr(g,"geoms",[]) if isinstance(a,(Polygon,MultiPolygon)) and not a.is_empty]
    return unary_union(parts) if parts else Polygon()

def read_projected(path):
    g=gpd.read_file(path)
    g=g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy().reset_index(drop=True)
    if g.crs is None:
        b,rb=g.total_bounds,tuple(BOUNDS)
        if not (b[2]>=rb[0] and b[0]<=rb[2] and b[3]>=rb[1] and b[1]<=rb[3]):
            raise ValueError(f"Missing CRS and non-overlapping XY coordinates: {path}")
        print("Assigning Baego raster CRS:",path.name)
        g=g.set_crs(CRS)
    else:g=g.to_crs(CRS)
    g["geometry"]=gpd.GeoSeries([polygonal(x) for x in g.geometry],index=g.index,crs=CRS)
    return g.loc[~g.geometry.is_empty].reset_index(drop=True)

roi_gdf=read_projected(ROI_FILE)
assert len(roi_gdf)==6
reference=read_projected(REF_FILE)
reference=reference.loc[reference.geometry.area>=REFERENCE_MIN_AREA].reset_index(drop=True)
predicted={i:read_projected(SAM_DIR/f"roi{i}_{METHOD}.gpkg") for i in ROIS}
print("Six Baego ROIs; reference crowns:",len(reference))
display(pd.DataFrame([
    {"ROI":i,"roi_ha":roi_gdf.geometry.iloc[i].area/1e4,
     "reference_centroids":int(reference.geometry.centroid.within(roi_gdf.geometry.iloc[i]).sum()),
     "predicted_centroids":int(predicted[i].geometry.centroid.within(roi_gdf.geometry.iloc[i]).sum())}
    for i in ROIS]))

## 03 — Fixed, non-overlapping evaluation windows and coverage audit

Generate **full** square windows per ROI. Grid boundaries are anchored to a fixed global metric grid (`floor(x / size) * size`) rather than the SAM inference core grid. Retain only cells with ≥99.9% geometric intersection with the ROI. This avoids confusing a small clipped fragment with a true 40 × 40 m sampling unit. It may exclude edge areas; the whole-ROI result is also evaluated separately.

Window counts decline with scale. **If 80 m windows are unavailable, report this rather than inventing larger coverage**. Windows at different scales are nested/correlated, not independent replicates. Compare scale summaries using ROIs as the replication units, not individual window counts.

In [ ]:
# 03 — Non-overlapping squares per scale; measure sampling footprint
def windows_for_region(region,scale):
    xmin,ymin,xmax,ymax=region.bounds
    xs=range(int(np.floor(xmin/scale)),int(np.ceil(xmax/scale)))
    ys=range(int(np.floor(ymin/scale)),int(np.ceil(ymax/scale)))
    cells=[]
    for ix in xs:
        for iy in ys:
            square=box(ix*scale,iy*scale,(ix+1)*scale,(iy+1)*scale)
            fraction=square.intersection(region).area/(scale*scale)
            if fraction >= MIN_INSIDE_FRACTION:
                cells.append(square)
    return cells

WINDOWS=[]
for roi in ROIS:
    region=roi_gdf.geometry.iloc[roi]
    for scale in SCALES_M:
        squares=windows_for_region(region,scale)
        for j,square in enumerate(squares):
            WINDOWS.append({"ROI":roi,"scale_m":scale,"window_id":j,
                            "area_m2":square.area,"geometry":square})
WINDOWS_GDF=gpd.GeoDataFrame(WINDOWS,geometry="geometry",crs=CRS)
footprint=(WINDOWS_GDF.groupby(["ROI","scale_m"]).agg(
    n_windows=("window_id","size"),area_m2=("area_m2","sum")).reset_index())
allkeys=pd.MultiIndex.from_product([ROIS,SCALES_M],names=["ROI","scale_m"]).to_frame(index=False)
footprint=allkeys.merge(footprint,on=["ROI","scale_m"],how="left").fillna({"n_windows":0,"area_m2":0})
footprint["roi_ha"]=footprint.ROI.map({i:roi_gdf.geometry.iloc[i].area/1e4 for i in ROIS})
footprint["fraction_roi_sampled"]=footprint.area_m2/(footprint.roi_ha*10000)
print("=== Evaluation geometry coverage; missing 80m cells remain explicitly zero ===")
display(footprint.round(3))
footprint.to_csv(OUT/"04_window_coverage.csv",index=False)

## 04 — Ecological error at each window scale

For every window, calculate reference and SAM centroid counts, density, median/mean crown area, clipped union canopy cover and area IoU. Compute **signed cover bias (percentage points)**, **absolute cover error**, **reference-only** and **SAM-only** area percentages, and their sum (total spatial disagreement).

A key diagnostic is the **cancellation ratio**: how much of the gross omission-plus-commission disagreement disappears when only net cover bias is reported. Canopy IoU uses exact vector polygons.

Crown counts can be zero in small windows: preserve count errors, but do not compute undefined relative density errors. Median crown area requires enough reference crowns for interpretation.

In [ ]:
# 04 — Accurate vector area and crown metrics, indexed by spatial window
def union_clipped(geoms, region):
    clipped=[polygonal(g.intersection(region)) for g in geoms if g.intersects(region)]
    clipped=[g for g in clipped if not g.is_empty and g.area>0]
    return polygonal(unary_union(clipped)) if clipped else Polygon()

def metric_pair(roi_id,region,scale_m,window_id):
    ref=reference
    sam=predicted[roi_id]
    ref_in=ref.loc[ref.geometry.centroid.within(region)]
    sam_in=sam.loc[sam.geometry.centroid.within(region)]
    nref,nsam=len(ref_in),len(sam_in)
    ref_geom=union_clipped(ref.loc[ref.geometry.intersects(region)].geometry,region)
    sam_geom=union_clipped(sam.loc[sam.geometry.intersects(region)].geometry,region)
    common=ref_geom.intersection(sam_geom).area
    ref_area,sam_area=ref_geom.area,sam_geom.area
    region_area=region.area
    ref_only=(ref_area-common)/region_area*100
    sam_only=(sam_area-common)/region_area*100
    signed=(sam_area-ref_area)/region_area*100
    gross=ref_only+sam_only
    ref_sizes=ref_in.geometry.area.to_numpy()
    sam_sizes=sam_in.geometry.area.to_numpy()
    can_union=ref_area+sam_area-common
    return {
        "ROI":roi_id,"scale_m":scale_m,"window_id":window_id,
        "window_ha":region_area/1e4,
        "ref_n":nref,"sam_n":nsam,
        "count_error":nsam-nref,
        "density_bias_pct":100*(nsam-nref)/nref if nref>0 else np.nan,
        "ref_mean_area_m2":float(np.mean(ref_sizes)) if nref else np.nan,
        "sam_mean_area_m2":float(np.mean(sam_sizes)) if nsam else np.nan,
        "ref_median_area_m2":float(np.median(ref_sizes)) if nref else np.nan,
        "ref_cv_area":float(np.std(ref_sizes,ddof=1)/np.mean(ref_sizes)) if nref>=3 else np.nan,
        "ref_canopy_pct":ref_area/region_area*100,
        "sam_canopy_pct":sam_area/region_area*100,
        "cover_bias_pp":signed,
        "abs_cover_error_pp":abs(signed),
        "sam_only_pp":sam_only,
        "ref_only_pp":ref_only,
        "gross_spatial_error_pp":gross,
        "cancellation_fraction":1-abs(signed)/gross if gross>0 else np.nan,
        "canopy_iou":common/can_union if can_union>0 else np.nan
    }

metric_rows=[]
for roi in ROIS:
    print("ROI",roi,"running window metrics...",flush=True)
    region=roi_gdf.geometry.iloc[roi]
    metric_rows.append(metric_pair(roi,region,0,"full_ROI"))
    for row in WINDOWS_GDF.loc[WINDOWS_GDF.ROI==roi].itertuples():
        metric_rows.append(metric_pair(roi,row.geometry,int(row.scale_m),int(row.window_id)))
METRICS=pd.DataFrame(metric_rows)
display(METRICS.loc[METRICS.scale_m.eq(0),
                    ["ROI","ref_n","sam_n","cover_bias_pp","canopy_iou","gross_spatial_error_pp","cancellation_fraction"]].round(3))
METRICS.to_csv(OUT/"04_window_ecological_metrics.csv",index=False)
print("Window rows:",len(METRICS),"saved:",OUT/"04_window_ecological_metrics.csv")

## 05 — Primary analysis: paired 10–20–40 m aggregation on identical footprints

This is the **main scale-sensitivity test**. Use only the 40 m windows confirmed fully inside the original ROI. Partition each of these *same* parent windows into four 20 m and sixteen 10 m cells. The parent and its subwindows therefore measure exactly the same ground area; no scale-dependent footprint selection is introduced.

For each parent, calculate the mean **absolute** canopy-cover error across subwindows, then compare with the absolute error of the aggregated parent. Report:

- paired 10 → 20 → 40 m mean absolute error (equal parent weight, then equal ROI weight);
- spatial disagreement between reference and SAM, which is additive across partitions and should remain constant on the same footprint;
- reduction in absolute error through cancellation (paired effect);
- within-parent canopy IoU and reference crown density.

**Important:** Decreasing absolute error under aggregation partly follows mathematically from averaging signed errors. The *magnitude* of cancellation and how it varies among forests are the ecological findings. The six ROI-level paired differences are the main replication units, not the 40 m parents or their subwindows.

In [ ]:
# 05 — Strict common-footprint paired aggregation; no new SAM inference
PARENTS = WINDOWS_GDF.loc[WINDOWS_GDF.scale_m.eq(40)].copy()
if PARENTS.empty:
    raise RuntimeError("No complete 40 m parent windows; cannot construct paired analysis")
def partition_square(parent, child_size):
    xmin,ymin,xmax,ymax = parent.bounds
    width=xmax-xmin
    n=int(round(width/child_size))
    if not np.isclose(n*child_size,width):
        raise ValueError("Parent window cannot be partitioned evenly")
    return [box(xmin+ix*child_size,ymin+iy*child_size,
                xmin+(ix+1)*child_size,ymin+(iy+1)*child_size)
            for ix in range(n) for iy in range(n)]

paired_rows=[]
for p in PARENTS.itertuples():
    region=p.geometry
    roi=int(p.ROI)
    parent_key=f"ROI{roi}_P{int(p.window_id):03d}"
    for size in (10,20,40):
        children=partition_square(region,size)
        if len(children)!=(40//size)**2:
            raise AssertionError("Unexpected number of subwindows")
        m=[metric_pair(roi,child,size,j) for j,child in enumerate(children)]
        signed=np.array([x["cover_bias_pp"] for x in m],dtype=float)
        gross=np.array([x["gross_spatial_error_pp"] for x in m],dtype=float)
        ious=np.array([x["canopy_iou"] for x in m],dtype=float)
        coverage=np.array([x["ref_canopy_pct"] for x in m],dtype=float)
        paired_rows.append({
            "ROI":roi,"parent_id":parent_key,"scale_m":size,
            "n_subwindows":len(children),
            "parent_area_m2":region.area,
            "mean_abs_cover_error_pp":float(np.mean(np.abs(signed))),
            "mean_signed_cover_bias_pp":float(np.mean(signed)),
            "gross_disagreement_pp":float(np.mean(gross)),
            "mean_subwindow_canopy_iou":float(np.nanmean(ious)),
            "mean_ref_cover_pct":float(np.mean(coverage)),
            "ref_parent_crowns":int(metric_pair(roi,region,40,"p")["ref_n"])
        })
PAIRED=pd.DataFrame(paired_rows)
# Within each parent all aggregation levels cover identical footprints.
check=PAIRED.pivot(index=["ROI","parent_id"],columns="scale_m",
                   values=["mean_signed_cover_bias_pp","gross_disagreement_pp"])
for variable in ("mean_signed_cover_bias_pp","gross_disagreement_pp"):
    values=check[variable].to_numpy()
    if not np.allclose(values,values[:,[0]],rtol=0,atol=1e-6):
        raise AssertionError(f"{variable} changed under exact aggregation: check partition boundaries")
PAIRED["error_reduction_vs_10m_pp"]=PAIRED.groupby(
    ["ROI","parent_id"])["mean_abs_cover_error_pp"].transform("first") - PAIRED.mean_abs_cover_error_pp
# Explicit reference to the 10m row avoids relying on incidental group order.
baseline=PAIRED.loc[PAIRED.scale_m.eq(10),["parent_id","mean_abs_cover_error_pp"]].rename(
    columns={"mean_abs_cover_error_pp":"baseline_10m_mae"})
PAIRED=PAIRED.drop(columns=["error_reduction_vs_10m_pp"]).merge(baseline,on="parent_id",validate="many_to_one")
PAIRED["error_reduction_vs_10m_pp"]=PAIRED.baseline_10m_mae-PAIRED.mean_abs_cover_error_pp
# Quantify ROI-level means first; parent squares are spatially correlated.
PAIRED_ROI=(PAIRED.groupby(["ROI","scale_m"],as_index=False)
            .agg(n_parents=("parent_id","nunique"),
                 mean_abs_cover_error_pp=("mean_abs_cover_error_pp","mean"),
                 mean_error_reduction_pp=("error_reduction_vs_10m_pp","mean"),
                 mean_gross_disagreement_pp=("gross_disagreement_pp","mean")))
PAIRED_SUMMARY=(PAIRED_ROI.groupby("scale_m",as_index=False)
                .agg(n_rois=("ROI","nunique"),
                     n_parents=("n_parents","sum"),
                     mean_abs_cover_error_pp=("mean_abs_cover_error_pp","mean"),
                     sd_between_roi=("mean_abs_cover_error_pp","std"),
                     mean_reduction_vs_10m_pp=("mean_error_reduction_pp","mean"),
                     mean_gross_disagreement_pp=("mean_gross_disagreement_pp","mean")))
display(PAIRED_SUMMARY.round(3))
print("=== Paired ROI means (each ROI is one interpretive replicate) ===")
display(PAIRED_ROI.round(3))
for df,name in [(PAIRED,"04_paired_parent_scale_metrics.csv"),
                (PAIRED_ROI,"04_paired_by_roi.csv"),
                (PAIRED_SUMMARY,"04_paired_scale_summary.csv")]:
    df.to_csv(OUT/name,index=False)

fig,ax=plt.subplots(figsize=(9,6))
for roi,grp in PAIRED_ROI.groupby("ROI"):
    ax.plot(grp.scale_m,grp.mean_abs_cover_error_pp,marker="o",
            alpha=.6,label=f"ROI {roi}")
ax.plot(PAIRED_SUMMARY.scale_m,PAIRED_SUMMARY.mean_abs_cover_error_pp,
        color="black",linewidth=3,marker="s",label="Equal-ROI mean")
ax.set(xlabel="Aggregation window width (m)",
       ylabel="Mean absolute canopy-cover error (percentage points)",
       title="Paired multiscale test: identical 40 m parent footprints",
       xticks=[10,20,40])
ax.legend(ncol=2,fontsize=8);ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT/"04_paired_scale_dependence.png",dpi=200,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(9,5))
ax.plot(PAIRED_SUMMARY.scale_m,PAIRED_SUMMARY.mean_abs_cover_error_pp,
        marker="o",label="Mean absolute cover error")
ax.plot(PAIRED_SUMMARY.scale_m,PAIRED_SUMMARY.mean_gross_disagreement_pp,
        marker="s",label="Gross spatial disagreement")
ax.set(xlabel="Aggregation window width (m)",
       ylabel="Area (%) / cover error (percentage points)",
       title="Cancellation without improved spatial delineation",
       xticks=[10,20,40])
ax.legend();ax.grid(alpha=.2);fig.tight_layout()
fig.savefig(OUT/"04_paired_error_cancellation.png",dpi=200,bbox_inches="tight")
plt.show()
print("Paired common-footprint analysis saved. Whole-ROI results are not paired.")

## 06 — Reference-only forest-structure predictors at 20 and 40 m

Estimate structure independently from the SAM predictions: centroid-selected reference crown density, crown-size median/CV and **nearest-neighbour crown spacing**, computed using the full set of crowns in each ROI to reduce artificial window-edge crowding effects. The reference union canopy cover is also included, but its relationship with cover error is potentially mathematically coupled.

The main explanatory analysis uses 20 m windows (more spatial support); 40 m is a sensitivity check. For each, report window counts and between-ROI variation. A window with very few crowns cannot yield reliable crown-size heterogeneity.

Our six ROIs are **not six independent site replicates**. Structural correlations are descriptive and should not be represented as definitive tests of causal forest effects.

In [ ]:
# 06 — Reference-only crowding and heterogeneity predictors
crowd_maps={}
for roi in ROIS:
    region=roi_gdf.geometry.iloc[roi]
    selected=reference.loc[reference.geometry.centroid.within(region)]
    xy=np.array([[p.x,p.y] for p in selected.geometry.centroid],dtype=float).reshape(-1,2)
    if len(xy)>=2:
        tree=cKDTree(xy);d,_=tree.query(xy,k=2)
        crowd_maps[roi]=(selected.index.to_numpy(),d[:,1])
    else:
        crowd_maps[roi]=(selected.index.to_numpy(),np.full(len(xy),np.nan))
crowd_series={roi:pd.Series(dist,index=ind) for roi,(ind,dist) in crowd_maps.items()}
def local_crowding(roi,region):
    selected=reference.loc[reference.geometry.centroid.within(region)]
    vals=crowd_series[roi].reindex(selected.index).dropna()
    return float(vals.mean()) if len(vals) else np.nan
crowd_rows=[]
for row in METRICS.itertuples():
    roi=int(row.ROI)
    if row.scale_m==0:geometry=roi_gdf.geometry.iloc[roi]
    else:
        geometry=WINDOWS_GDF.loc[
            (WINDOWS_GDF.ROI==roi)&(WINDOWS_GDF.scale_m==row.scale_m)&
            (WINDOWS_GDF.window_id==row.window_id),"geometry"].iloc[0]
    crowd_rows.append(local_crowding(roi,geometry))
METRICS["ref_mean_NN_distance_m"]=crowd_rows
METRICS["ref_density_ha"]=METRICS.ref_n/METRICS.window_ha
METRICS.to_csv(OUT/"04_window_ecological_structure_metrics.csv",index=False)
show=METRICS.loc[METRICS.scale_m.eq(40),[
    "ROI","ref_n","ref_density_ha","ref_median_area_m2","ref_cv_area",
    "ref_mean_NN_distance_m","ref_canopy_pct","abs_cover_error_pp"]]
print("=== Reference forest structure and cover error at 40 m ===")
display(show.head(20).round(2))

## 07 — Forest structure versus ecological errors: 20 m primary, 40 m sensitivity

Use the reference-only covariates from Section 06 to examine ecological errors at 20 m and 40 m. Visualize both absolute canopy-cover error and crown-density bias, where defined. Include ROI-specific points and summaries rather than pooling thousands of windows as if independent.

Assess crown-level context as a **separate** outcome using the optional saved Notebook 03 `03_section09_crown_coverage_and_merges.csv` and `03_reference_crowns_probability_and_matching.csv`. This allows testing whether local object matching and actual canopy omission behave differently from area-based cover bias. Do not infer species effects from the U-Net probability map alone.

In [ ]:
# 07 — Descriptive within-ROI structure analysis at 20m and 40m
predictors=[
    ("ref_density_ha","Crown density (ha⁻¹)"),
    ("ref_median_area_m2","Median crown area (m²)"),
    ("ref_cv_area","Crown-area CV"),
    ("ref_mean_NN_distance_m","Mean nearest-neighbour spacing (m)"),
    ("ref_canopy_pct","Reference canopy cover (%)")]
associations=[]
for FIXED_SCALE in (20,40):
    D=METRICS.loc[METRICS.scale_m.eq(FIXED_SCALE)].copy()
    fig,axes=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
    for ax,(column,label) in zip(axes.flat,predictors):
        subset=D.dropna(subset=[column,"abs_cover_error_pp"])
        for roi,g in subset.groupby("ROI"):
            ax.scatter(g[column],g.abs_cover_error_pp,
                       s=18,alpha=.6,label=f"ROI {roi}")
        if len(subset)>=5 and subset[column].nunique()>2:
            cx=subset[column]-subset.groupby("ROI")[column].transform("mean")
            cy=subset.abs_cover_error_pp-subset.groupby("ROI").abs_cover_error_pp.transform("mean")
            if cx.nunique()>2 and cy.nunique()>2:
                rho=float(spearmanr(cx,cy).statistic)
            else:rho=np.nan
        else:rho=np.nan
        associations.append({"scale_m":FIXED_SCALE,"predictor":column,
                             "n_windows":len(subset),"n_rois":subset.ROI.nunique(),
                             "ROI_demeaned_rank_correlation":rho})
        ax.set(xlabel=label,ylabel="Absolute canopy-cover error (pp)")
        ax.grid(alpha=.16)
    axes.flat[5].axis("off")
    axes.flat[0].legend(fontsize=8,ncol=2)
    fig.suptitle(f"Exploratory forest-structure associations: {FIXED_SCALE} m windows")
    fig.savefig(OUT/f"04_structure_dependence_{FIXED_SCALE}m.png",dpi=180,bbox_inches="tight")
    plt.show()
ASSOCIATIONS=pd.DataFrame(associations)
print("=== Descriptive forest-structure associations, without naive p-values ===")
display(ASSOCIATIONS.round(3))
ASSOCIATIONS.to_csv(OUT/"04_structure_20_40m_exploratory.csv",index=False)

# Additional ecological outcome: signed crown density bias, when reference n>0.
for scale in (20,40):
    subset=METRICS.loc[(METRICS.scale_m==scale)&METRICS.density_bias_pct.notna()]
    print(f"{scale}m: {len(subset)} windows with defined relative density bias; "
          f"median bias {subset.density_bias_pct.median():.1f}%")

# Optional independent object-level structural evidence from NB03.
nb03=ROOT/"V2/Crown/musanga_sam1_diagnostics"
f_crown=nb03/"03_reference_crowns_probability_and_matching.csv"
f_coverage=nb03/"03_section09_crown_coverage_and_merges.csv"
if f_crown.exists() and f_coverage.exists():
    a=pd.read_csv(f_crown)
    b=pd.read_csv(f_coverage)
    combined=a.merge(b,on=["ROI","ref_local_id"],validate="one_to_one")
    combined["size_tertile"]=pd.qcut(combined.ref_area_m2,q=3,
                                   labels=["small","medium","large"])
    crown_scale=combined.groupby("size_tertile",observed=True).agg(
        n=("matched_iou_ge_0_5","size"),
        match_rate=("matched_iou_ge_0_5","mean"),
        near_omission_fraction=("candidate_omission_lt_10pct","mean"),
        multicrown_fraction=("associated_with_multicrown_sam","mean"),
        median_area_m2=("ref_area_m2","median")).reset_index()
    print("=== Crown-level structural context (separate from window metrics) ===")
    display(crown_scale.round(3))
    crown_scale.to_csv(OUT/"04_crown_size_matching_context.csv",index=False)
else:
    print("NB03 crown-level CSVs absent; skipping optional crown context, not replacing them.")

## 08 — Manuscript-ready interpretation and safeguards

**Primary result:** the strictly paired 10, 20 and 40 m comparison from Section 05. Main endpoint: mean absolute canopy-cover error calculated at each aggregation scale on the **identical 40 m parent footprint**. Report six equal-weight ROI-level estimates, associated between-ROI spread and the number of parent windows. Gross spatial disagreement should remain constant across partition scales by construction.

**Secondary result:** descriptive 20 m forest-structure relationships, with 40 m sensitivity, based on reference crown density, crown-size median/CV, spacing and canopy cover. Object-level size/matching findings from Notebook 03 supplement but do not substitute for this analysis.

**Other scales:** earlier independent 10–80 m windows remain available as a **coverage audit**, not the primary proof of aggregation effects. 80 m windows are missing for three ROIs. The whole-ROI estimates (~2 ha) use a different extent and should appear only as contextual endpoints, not fitted onto a continuous scale regression.

**Constraints to address before publication:** (1) only six ROIs and local spatial dependence; (2) harmonize stand-age identifiers with Nadine's seven-stand chronosequence; (3) confirm Kristof annotation completeness and role of other annotators/QField; (4) replicate the spatial analysis with the exact final SAM-frst masks (RGB/height/SLIC), not just frozen RGB SAM1 NMS; (5) crown matching IoU thresholds and reference definition differ between experiments.

**Potential result heading:** *Ecological reliability is scale dependent: spatial omission and commission errors compensate when canopy estimates are aggregated.*

No p-values or claims about generalized model transferability are justified from the six exploratory ROIs alone.

In [ ]:
# 08 — Concise manuscript discussion tables
print("Primary paired-scale results (the same ground footprints):")
display(PAIRED_SUMMARY.round(3))
print("Forest-structure 20/40m descriptive relationships:")
display(ASSOCIATIONS.round(3))
print("Whole-ROI results, contextual (not strictly paired):")
display(METRICS.loc[METRICS.scale_m.eq(0),
    ["ROI","ref_n","sam_n","density_bias_pct","cover_bias_pp",
     "canopy_iou","gross_spatial_error_pp","cancellation_fraction"]].round(3))
print("Saved in:",OUT)
print("Frozen SAM1 predictions and annotations unchanged.")